Installing the dependencies


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [ ]:
import os
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "google/gemma-2-2b-it",
    dtype = None, # None for auto detection
    max_seq_length = 1024, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # Set to False to enable LoRA and save memory
    token = os.environ["HF_TOKEN"], # use one if using gated models
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files='/content/final_merged_dataset.json')
print(dataset)

In [ ]:
print(dataset['train'][0])

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset["train"])


In [ ]:
dataset

In [ ]:
dataset[1]

In [ ]:
from trl import SFTTrainer, SFTConfig

alpaca_prompt = '''Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}'''

def formatting_func(examples):
    texts = []
    # Determine if the input is batched (values are lists) or a single example (values are single items)
    if isinstance(examples['question'], list):
        questions = examples['question']
        answers = examples['answer']
    else:
        questions = [examples['question']] # Wrap single item in a list
        answers = [examples['answer']]     # Wrap single item in a list

    for question, answer in zip(questions, answers):
        formatted_text = alpaca_prompt.format(question, "", answer) + tokenizer.eos_token
        texts.append(formatted_text)
    return texts # Return the list of processed strings directly, as per Unsloth's error message

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    formatting_func = formatting_func, # Pass the formatting function here
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
if True:
    model.push_to_hub_merged(
        "Shiva-k22/gemma-FinAI", tokenizer,
        token = "YOUR-HUGGINGFACE-TOKEN"
    )

In [ ]:
if True:
    model.push_to_hub_gguf(
        "Shiva-k22/gemma-FinAI",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR-HUGGINGFACE-TOKEN",
    )